So far we've treated code as sequential operations. Modern computers, however, allow for different levels of parallel code execution.

For decades, until the year 2000, CPU core performance improved rapidly as transistors became smaller and allowed for clock frequencies to grow exponentially. After 2000, this scaling relationship broke down because of physical limitations. This is one of the reasons why CPU manufacturers started making *multicore* CPUs. Consumer CPUs may have up to tens of cores, whereas server CPUs can have hundreds.

![After 2000, clock frequencies stopped growing and multicore CPUs emerged (from: https://en.wikipedia.org/wiki/Clock_rate)](images/CPU_clock_speed_and_Core_count_Graph.png)

## Multiprocessing

The easiest way of taking advantage of multicore CPUs is by running multiple programs, or processes, in parallel. While your image processing pipeline is running, fully utilizing one of the CPU cores, you can still edit your Word documents, browse the internet, and listen to a podcast all at the same time.

That's convenient, but this also means that it could be beneficial to start two or more image processing pipelines at the same time, each processing a subset of your dataset. This is the simplest form of *multiprocessing*, i.e. running multiple processes in parallel.

Multiple *worker* processes can also be initiated from one parent process. This means, for example, that your Python code can start multiple worker Python processes that perform their own task. The built-in module `multiprocessing`, and the package [joblib](https://joblib.readthedocs.io/) allow you to do this quite conveniently. Although `joblib` is not built-in, it has the advantage that it works flawlessly from Jupyter and on Windows. The following example starts 3 worker processes that each execute `f(x)` for a different `x`. In this example, the function `f()` returns the process ID of the worker and x²:

In [2]:
import os
from joblib import Parallel, delayed


def f(x):
    """
    Returns the process ID and x²
    """
    return os.getpid(), x * x


# Execute f(x) for x = 0 ... 9 using 3 workers.
result = Parallel(n_jobs=3)(
    delayed(f)(x) for x in range(10)
)

print("Main process ID:", os.getpid())
print("Results:", result)

Main process ID: 28292
Results: [(24640, 0), (27776, 1), (34704, 4), (24640, 9), (24640, 16), (24640, 25), (27776, 36), (24640, 49), (27776, 64), (24640, 81)]


 As a rule of thumb, start with a low number of workers (e.g. 3 or 4) and benchmark the performance. Too many workers compete for CPU time, potentially slowing everything down. And besides competing for CPU time, each process has its own memory overhead; which means that starting too many workers may also exhaust RAM memory.

For small tasks, starting and coordinating workers can cost more time than the computation saves. This means that multiprocessing should only be used for parallelizing tasks that take several seconds to complete.

## Multithreading

A single process can make use of multiple CPU cores too. This technique is called *multithreading*. Multithreading takes away the overhead of starting workers and allows sharing memory between threads.

Python multithreading behaves differently from languages such as C++ or Java for CPU-intensive work. In standard Python, only one thread at a time can execute plain Python code within a process. This means that although we can start multiple parallel threads from Python, the interpreter will not execute any Python code in parallel. Compiled code, on the other hand, *can* be executed in parallel by starting multiple threads. We could start multiple threads from within our C by adding [OpenMP compiler pragmas](https://en.wikipedia.org/wiki/OpenMP), or we could use `joblib` to start multiple threads. `joblib` also has specific optimizations for NumPy arrays.

The example below shows how to use `joblib` for calculating the Fourier transform of 12 images with random numbers using multithreading:

In [154]:
from time import perf_counter

import numpy as np


# Generate 12 499×499 images with random numbers.
images = [np.random.randn(499, 499) for _ in range(12)]

# Apply a fast Fourier transform to all 12 images.
start = perf_counter()
results = [np.fft.fft(img) for img in images]
print(f"Elapsed time without multithreading: {perf_counter() - start:.3f} seconds")

# Apply a fast Fourier transform to all 12 images.
# Use joblib multithreading with 4 threads.
start = perf_counter()
results = Parallel(n_jobs=4, prefer="threads")(
    delayed(np.fft.fft)(img) for img in images
)
print(f"Elapsed time with multithreading: {perf_counter() - start:.3f} seconds")

Elapsed time without multithreading: 0.153 seconds
Elapsed time with multithreading: 0.060 seconds


A similar disclaimer as above: for very small tasks, starting and coordinating threads can cost more time than the computation saves.

## Loop Vectorization

This is perhaps the least well known, but a very effective way of parallelizing computations. A single CPU core can perform the same operation on several numbers simultaneously. This is known as *Single Instruction, Multiple Data*, or SIMD. Whereas multithreading and multiprocessing both parallelize work by dividing different tasks between CPU cores (known as *Multiple Instruction, Multiple Data*, or MIMD), loop vectorization parallelizes a single instruction applied to multiple data on a single CPU core. For example, instead of multiplying one floating point number at a time, a CPU core can multiply 4, 8, or even more numbers in a single instruction. This means that loop vectorization can be used in addition to either multithreading or multiprocessing!

A CPU core has special registers and execution units for vector instructions. A 256-bit vector register can hold 4 64-bit or 8 32-bit floating point numbers. When a compiler determines that multiple iterations of a for-loop can be executed together safely and efficiently using SIMD instructions, it can automatically vectorize that for-loop. This is usually the case for simple for-loops, i.e. loops with only regular arithmetic operations. With the `gcc` compiler, loop vectorization can be enabled using the `-ftree-vectorize` flags. When you've written an image processing algorithm in a compiled language such as C or C++, it is often worthwhile to try enabling loop vectorization.

## Parallelism: CPU vs GPU

SIMD is also one of the key ideas behind GPU computing, but GPUs typically use a related execution model called SIMT (Single Instruction, Multiple Threads). SIMT allows groups of GPU threads to perform the same instruction on their own data, with some ability to diverge.

> **External course material**
>
> The following YouTube video explains about different kinds of parallelism and the differences between CPU and GPU computing:
>
> * <https://www.youtube.com/watch?v=kbWKOSVHfvc>

## Assignments
You've reached the end of this week's study material! During Friday's practical session, we'll continue working on the third graded assignment. This group assignment is handed out on Friday.